# Ejercicio 2 - Listado 1 del tema 5

Se va a probar a crear una arquitectura transformer tipo encoder desde cero para identificar fake news en español. 

Se piden dos versiones del ejercicio:

a) Sin utilizar positional embeddings. 

b) Utilizando positional embeddings. 

Hay que analizar los resultados de ambos casos y contrastarlos. En este notebook se comparan ambas versiones a través del código. 


In [1]:
import os
import random
import numpy as np
import pandas as pd
import tensorflow as tf
from keras_nlp.layers import PositionEmbedding
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Embedding, Dense, Dropout, GlobalAveragePooling1D, LayerNormalization, MultiHeadAttention
from tensorflow.keras.optimizers import Adam

In [2]:
# =====================
# 1. Cargar y preparar datos
# =====================
# ======= 1. Reproducibilidad ========
SEED = 42
os.environ['PYTHONHASHSEED'] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

# ======= 2. Datos ========
df = pd.read_excel("../DataSets/FakeNewsCorpusSpanish/train.xlsx", engine="openpyxl")
df.head()

texts = df["Text"].astype(str).tolist()
labels = df["Category"].astype(str).tolist()

label_encoder = LabelEncoder()
labels = label_encoder.fit_transform(labels)

In [3]:
# =====================
# 2. División del dataset
# =====================
X_train_texts, X_temp_texts, y_train, y_temp = train_test_split(texts, labels, test_size=0.2, random_state=SEED)
X_val_texts, X_test_texts, y_val, y_test = train_test_split(X_temp_texts, y_temp, test_size=0.5, random_state=SEED)

In [4]:
# =====================
# 3. Tokenización y padding
# =====================
VOCAB_SIZE = 10000
MAX_LEN = 200

tokenizer = Tokenizer(num_words=VOCAB_SIZE, oov_token="<OOV>")
tokenizer.fit_on_texts(X_train_texts)

X_train_seq = tokenizer.texts_to_sequences(X_train_texts)
X_val_seq = tokenizer.texts_to_sequences(X_val_texts)
X_test_seq = tokenizer.texts_to_sequences(X_test_texts)

X_train = pad_sequences(X_train_seq, maxlen=MAX_LEN)
X_val = pad_sequences(X_val_seq, maxlen=MAX_LEN)
X_test = pad_sequences(X_test_seq, maxlen=MAX_LEN)

In [5]:
# ======= 4. Transformer Encoder Block ========
def transformer_encoder(inputs, head_size, num_heads, ff_dim, dropout=0.1):
    x = MultiHeadAttention(num_heads=num_heads, key_dim=head_size)(inputs, inputs)
    x = Dropout(dropout)(x)
    x = LayerNormalization(epsilon=1e-6)(x + inputs)
    
    ff = Dense(ff_dim, activation="relu")(x)
    ff = Dense(inputs.shape[-1])(ff)
    ff = Dropout(dropout)(ff)
    return LayerNormalization(epsilon=1e-6)(x + ff)

In [6]:
# =====================
# 5. Construcción del modelo 
# =====================
def build_model(use_positional_embedding):
    inputs = Input(shape=(MAX_LEN,), dtype="int32")
    
    x = Embedding(input_dim=VOCAB_SIZE, output_dim=64)(inputs)
    
    if use_positional_embedding:
        pos_x = PositionEmbedding(sequence_length=MAX_LEN)(x)
        x = x + pos_x
    
    x = transformer_encoder(x, head_size=64, num_heads=2, ff_dim=128)
    x = GlobalAveragePooling1D()(x)
    x = Dropout(0.1)(x)
    x = Dense(64, activation="relu")(x)
    x = Dropout(0.1)(x)
    outputs = Dense(1, activation="sigmoid")(x)
    
    model = Model(inputs, outputs)
    model.compile(optimizer=Adam(1e-4), loss="binary_crossentropy", metrics=["accuracy"])
    model.summary()
    return model

In [7]:
# =====================
# 6. Entrenamiento de ambos modelos
# =====================
def train_and_evaluate(use_positional_embedding):
    name = "CON_POSITIONAL" if use_positional_embedding else "SIN_POSITIONAL"
    print(f"\n Entrenando modelo: {name}\n")
    
    model = build_model(use_positional_embedding)
    model.fit(X_train, y_train, validation_data=(X_val, y_val), epochs=5, batch_size=32, verbose=2)
    
    y_pred_probs = model.predict(X_test)
    y_pred = (y_pred_probs > 0.5).astype(int).reshape(-1)

    print(f"\n Reporte para {name}")
    print(classification_report(y_test, y_pred, target_names=label_encoder.classes_))
    return model

In [8]:
# =====================
# 7. Evaluación final
# =====================
model_with_pos = train_and_evaluate(use_positional_embedding=True)
model_without_pos = train_and_evaluate(use_positional_embedding=False)


 Entrenando modelo: CON_POSITIONAL



Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                  ┃ Output Shape              ┃         Param # ┃ Connected to               ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)      │ (None, 200)               │               0 │ -                          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ embedding (Embedding)         │ (None, 200, 64)           │         640,000 │ input_layer[0][0]          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ position_embedding            │ (None, 200, 64)           │          12,800 │ embedding[0][0]            │
│ (PositionEmbedding)           │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ add (Add)                     │ (None, 200, 64)           │               0 │ embedding[0][0],           │
│                               │                           │                 │ position_embedding[0][0]   │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ multi_head_attention          │ (None, 200, 64)           │          33,216 │ add[0][0], add[0][0]       │
│ (MultiHeadAttention)          │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ dropout_1 (Dropout)           │ (None, 200, 64)           │               0 │ multi_head_attention[0][0] │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ add_1 (Add)                   │ (None, 200, 64)           │               0 │ dropout_1[0][0], add[0][0] │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ layer_normalization           │ (None, 200, 64)           │             128 │ add_1[0][0]                │
│ (LayerNormalization)          │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ dense (Dense)                 │ (None, 200, 128)          │           8,320 │ layer_normalization[0][0]  │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ dense_1 (Dense)               │ (None, 200, 64)           │           8,256 │ dense[0][0]                │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ dropout_2 (Dropout)           │ (None, 200, 64)           │               0 │ dense_1[0][0]              │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ add_2 (Add)                   │ (None, 200, 64)           │               0 │ layer_normalization[0][0], │
│                               │                           │                 │ dropout_2[0][0]            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ layer_normalization_1         │ (None, 200, 64)           │             128 │ add_2[0][0]                │
│ (LayerNormalization)          │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ global_average_pooling1d      │ (None, 64)                │               0 │ layer_normalization_1[0][… │
│ (GlobalAveragePooling1D)      │                           │               

 Total params: 707,073 (2.70 MB)

 Trainable params: 707,073 (2.70 MB)

...

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                  ┃ Output Shape              ┃         Param # ┃ Connected to               ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ input_layer_1 (InputLayer)    │ (None, 200)               │               0 │ -                          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ embedding_1 (Embedding)       │ (None, 200, 64)           │         640,000 │ input_layer_1[0][0]        │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ multi_head_attention_1        │ (None, 200, 64)           │          33,216 │ embedding_1[0][0],         │
│ (MultiHeadAttention)          │                           │                 │ embedding_1[0][0]          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ dropout_6 (Dropout)           │ (None, 200, 64)           │               0 │ multi_head_attention_1[0]… │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ add_3 (Add)                   │ (None, 200, 64)           │               0 │ dropout_6[0][0],           │
│                               │                           │                 │ embedding_1[0][0]          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ layer_normalization_2         │ (None, 200, 64)           │             128 │ add_3[0][0]                │
│ (LayerNormalization)          │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ dense_4 (Dense)               │ (None, 200, 128)          │           8,320 │ layer_normalization_2[0][… │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ dense_5 (Dense)               │ (None, 200, 64)           │           8,256 │ dense_4[0][0]              │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ dropout_7 (Dropout)           │ (None, 200, 64)           │               0 │ dense_5[0][0]              │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ add_4 (Add)                   │ (None, 200, 64)           │               0 │ layer_normalization_2[0][… │
│                               │                           │                 │ dropout_7[0][0]            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ layer_normalization_3         │ (None, 200, 64)           │             128 │ add_4[0][0]                │
│ (LayerNormalization)          │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ global_average_pooling1d_1    │ (None, 64)                │               0 │ layer_normalization_3[0][… │
│ (GlobalAveragePooling1D)      │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ dropout_8 (Dropout)           │ (None, 64)                │               0 │ global_average_pooling1d_… │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ dense_6 (Dense)               │ (None, 64)                │           4,160 │ dropout_8[0][0]            │
├───────────────────────────────┼───────────────────────────┼───────────────

 Total params: 694,273 (2.65 MB)

 Trainable params: 694,273 (2.65 MB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/5
17/17 - 7s - 412ms/step - accuracy: 0.5778 - loss: 0.6680 - val_accuracy: 0.6471 - val_loss: 0.6635
Epoch 2/5
17/17 - 1s - 74ms/step - accuracy: 0.6278 - loss: 0.6442 - val_accuracy: 0.6912 - val_loss: 0.6455
Epoch 3/5
17/17 - 1s - 68ms/step - accuracy: 0.6778 - loss: 0.6175 - val_accuracy: 0.6912 - val_loss: 0.6257
Epoch 4/5
17/17 - 1s - 67ms/step - accuracy: 0.7389 - loss: 0.5990 - val_accuracy: 0.7059 - val_loss: 0.6037
Epoch 5/5
17/17 - 1s - 72ms/step - accuracy: 0.7852 - loss: 0.5616 - val_accuracy: 0.6912 - val_loss: 0.5802
3/3 ━━━━━━━━━━━━━━━━━━━━ 1s 125ms/step

 Reporte para SIN_POSITIONAL
              precision    recall  f1-score   support

        Fake       0.71      0.83      0.77        36
        True       0.77      0.62      0.69        32

    accuracy                           0.74        68
   macro avg       0.74      0.73      0.73        68
weighted avg       0.74      0.74      0.73        68

